In [12]:
from dotenv import load_dotenv

load_dotenv()

True

In [13]:
from langchain_core.tools import tool


@tool
def publish_article(slug: str) -> str:
    """Publish an article to the web."""
    return f"✅ Published {slug}"


@tool
def list_articles() -> str:
    """List all articles (read-only)."""
    return "article-1\narticle-2\narticle-3"


print("Tools defined")

Tools defined


In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="claude-haiku-4-5",
    tools=[publish_article, list_articles],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"publish_article": {"allowed_decisions": ["approve", "reject"]}},
        ),
    ],
)

print("Agent with HITL middleware created")

Agent with HITL middleware created


In [15]:
# Test 1: Read-only (no interrupt)
print("\n" + "=" * 60)
print("TEST 1: List articles (no approval needed)")
print("=" * 60)

config = {"configurable": {"thread_id": "1"}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "List my articles"}]},
    config=config,
)

print(result["messages"][-1].content)


TEST 1: List articles (no approval needed)
You have 3 articles:

1. **article-1**
2. **article-2**
3. **article-3**

Would you like to publish any of these articles or do anything else with them?


In [16]:
# Test 2: Sensitive operation (interrupts)
print("\n" + "=" * 60)
print("TEST 2: Publish article (requires approval)")
print("=" * 60)

config = {"configurable": {"thread_id": "2"}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Publish article-1"}]},
    config=config,
)

if "__interrupt__" in result:
    desc = result["__interrupt__"][-1].value["action_requests"][-1]["description"]
    print(f"\n🔒 INTERRUPT: {desc}")


TEST 2: Publish article (requires approval)

🔒 INTERRUPT: Tool execution requires approval

Tool: publish_article
Args: {'slug': 'article-1'}


In [ ]:
from langgraph.types import Command

# Test 3: Approve and resume
print("\n" + "=" * 60)
print("TEST 3: Approve and resume")
print("=" * 60)

result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,  # Same thread
)

print(result["messages"][-1].content)

In [18]:
# Test 4: Reject request
print("\n" + "=" * 60)
print("TEST 4: Reject publication")
print("=" * 60)

config = {"configurable": {"thread_id": "3"}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Publish article-2"}]},
    config=config,
)

if "__interrupt__" in result:
    print("🔒 Approval required")
    result = agent.invoke(
        Command(resume={"decisions": [{"type": "reject", "message": "Not ready yet"}]}),
        config=config,
    )

print(result["messages"][-1].content)


TEST 4: Reject publication
🔒 Approval required
The article "article-2" is not ready to be published yet. It may still be in draft status or needs additional preparation before it can be published. You might want to check the article's status or complete any remaining work on it before attempting to publish.
